In [4]:
import zipfile
import os

# Unzip evons.zip
zip_path = "C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/evons.zip"
extract_path = "data/evons_csv"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("evons.csv extracted to:", extract_path)


evons.csv extracted to: data/evons_csv


In [5]:
import tarfile

# Function to extract tar.gz files
def extract_tar_gz(tar_path, extract_to):
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=extract_to)
    print(f"Extracted: {tar_path} → {extract_to}")

# Paths
images_tar = "C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/images.tar.gz"
articles_tar = "C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/articles.tar.gz"

# Extract locations
images_dir = "data/images"
articles_dir = "data/articles"

# Extract both
extract_tar_gz(images_tar, images_dir)
extract_tar_gz(articles_tar, articles_dir)


Extracted: C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/images.tar.gz → data/images
Extracted: C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/articles.tar.gz → data/articles


In [6]:
import pandas as pd
import os

# Path to the extracted CSV (adjust if needed)
csv_path = "data/evons_csv/evons.csv"  # or wherever you unzipped it

# Load the CSV
df = pd.read_csv(csv_path)

# Display original columns (optional)
print("Original Columns:", df.columns.tolist())


Original Columns: ['Unnamed: 0', 'article_fn', 'fb_comments', 'fb_likes', 'fb_shares', 'fb_engagements', 'media_source', 'is_fake', 'date', 'title', 'description', 'is_valid_image', 'image_fn', 'dominantColorForeground', 'dominantColorBackground', 'dominantColors', 'accentColor', 'isBwImg', 'image_tags', 'num_face', 'face_tags']


In [7]:
required_columns = ["description","title", "image_fn", "media_source", "is_fake"]
df_clean = df[required_columns].copy()

In [8]:
df_clean.dropna(subset=required_columns, inplace=True)

In [9]:
print("\nLabel Distribution:")
print(df_clean["is_fake"].value_counts())


Label Distribution:
is_fake
0    50144
1    42775
Name: count, dtype: int64


In [10]:
cleaned_csv_path = "data/evons_cleaned.csv"
df_clean.to_csv(cleaned_csv_path, index=False)

print(f"\n✅ Preprocessed CSV saved to: {cleaned_csv_path}")



✅ Preprocessed CSV saved to: data/evons_cleaned.csv


In [3]:
import pandas as pd
import os
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizer
from torchvision import transforms
from PIL import Image
import os

class FakeNewsDataset(Dataset):
    def __init__(self, dataframe, image_root_dir, max_length=128):
        self.df = dataframe.reset_index(drop=True)
        self.titles = dataframe["title"].fillna("").tolist()
        self.descriptions = dataframe["description"].fillna("").tolist()
        self.image_fns = dataframe["image_fn"].tolist()
        self.sources = dataframe["media_source"].tolist()
        self.labels = dataframe["is_fake"].tolist()
        self.image_root = image_root_dir
        self.max_length = max_length

        # Load tokenizer
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

        # Define image transform
        self.image_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Combine title and description
        text = self.titles[idx] + " [SEP] " + self.descriptions[idx]
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        # Build image path
        image_path = os.path.join(self.image_root, self.sources[idx], self.image_fns[idx])
        try:
            image = Image.open(image_path).convert("RGB")
            image = self.image_transform(image)
        except:
            image = torch.zeros((3, 224, 224))  # fallback

        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "image": image,
            "label": label
        }



In [4]:

from torch.utils.data import DataLoader

df = pd.read_csv("data/evons_cleaned.csv")
dataset = FakeNewsDataset(df, image_root_dir="data/images/images")
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)


In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer
from tqdm import tqdm

# Parameters
CSV_PATH = "data/evons_cleaned.csv"
MAX_LEN = 128
SAVE_PATH = "data/bert_tokenized.pt"

# Load cleaned CSV
df = pd.read_csv(CSV_PATH)

# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Containers
input_ids_list = []
attention_masks_list = []

print("🔁 Tokenizing rows...")
for i in tqdm(range(len(df))):
    title = str(df.loc[i, "title"])
    description = str(df.loc[i, "description"])
    combined = title + " [SEP] " + description

    encoding = tokenizer(
        combined,
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt'
    )

    input_ids_list.append(encoding['input_ids'].squeeze(0))
    attention_masks_list.append(encoding['attention_mask'].squeeze(0))

# Stack tensors
input_ids_tensor = torch.stack(input_ids_list)
attention_masks_tensor = torch.stack(attention_masks_list)

# Also get labels
labels_tensor = torch.tensor(df["is_fake"].tolist(), dtype=torch.long)

# Save all to .pt file
torch.save({
    "input_ids": input_ids_tensor,
    "attention_mask": attention_masks_tensor,
    "labels": labels_tensor
}, SAVE_PATH)

print(f"✅ BERT tokenization complete. Saved to: {SAVE_PATH}")


In [5]:
sample = dataset[0]
print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Image shape:", sample["image"].shape)
print("Label:", sample["label"])


Input IDs shape: torch.Size([128])
Attention mask shape: torch.Size([128])
Image shape: torch.Size([3, 224, 224])
Label: tensor(0)


In [6]:
sample = dataset[0]
if torch.equal(sample["image"], torch.zeros((3, 224, 224))):
    print("⚠️ Image was not found or failed to load.")
else:
    print("✅ Image loaded successfully.")


✅ Image loaded successfully.


In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

df = pd.read_csv("data/evons_cleaned.csv")
base_dir = "data/images/images"
broken_paths = []

def check_image(idx):
    try:
        row = df.loc[idx]
        image_path = os.path.join(base_dir, row['media_source'], row['image_fn'])
        Image.open(image_path).convert("RGB")
        return None  # valid
    except:
        return image_path  # broken

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = [executor.submit(check_image, i) for i in range(len(df))]
    for future in tqdm(as_completed(futures), total=len(df)):
        result = future.result()
        if result:
            broken_paths.append(result)

print(f"✅ Working: {len(df) - len(broken_paths)}")
print(f"❌ Broken: {len(broken_paths)}")


  7%|█████▌                                                                       | 6749/92919 [00:55<14:25, 99.54it/s]C:\Users\Shreyash Bhamare\anaconda3\envs\FKN\Lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 74%|███████████████████████████████████████████████████████▋                   | 69009/92919 [05:25<01:29, 266.62it/s]C:\Users\Shreyash Bhamare\anaconda3\envs\FKN\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
100%|███████████████████████████████████████████████████████████████████████████| 92919/92919 [09:03<00:00, 171.03it/s]

✅ Working: 87970
❌ Broken: 4949


In [8]:
df_broken = df[df.apply(lambda row: os.path.join("data/images/images", row["media_source"], row["image_fn"]) in broken_paths, axis=1)]
print(df_broken["is_fake"].value_counts())


is_fake
1    3348
0    1601
Name: count, dtype: int64


In [10]:
import os
import pandas as pd
from tqdm import tqdm

# Step 1: Load the CSV
csv_path = "data/evons_cleaned.csv"
base_image_dir = "data/images/images"
df = pd.read_csv(csv_path)

# Step 2: Build full image paths using vectorized zip
full_paths = [
    os.path.join(base_image_dir, src, fn)
    for src, fn in zip(df["media_source"], df["image_fn"])
]

# Step 3: Check if file exists (fast loop with tqdm)
tqdm.pandas(desc="🔍 Scanning images")
df["is_broken"] = [not os.path.exists(p) for p in tqdm(full_paths)]

# Step 4: Filter & Save
broken_rows = df[df["is_broken"]]
df_cleaned = df[~df["is_broken"]].drop(columns=["is_broken"])

# Save the filtered version
df_cleaned.to_csv("data/evons_cleaned_filtered.csv", index=False)

# Optional summary
print(f"✅ Saved {len(df_cleaned)} clean rows")
print(f"❌ Removed {len(broken_rows)} broken rows")


100%|██████████████████████████████████████████████████████████████████████████| 92919/92919 [00:20<00:00, 4463.30it/s]


✅ Saved 87971 clean rows
❌ Removed 4948 broken rows


In [11]:
# Load the filtered dataset
filtered_path = "data/evons_cleaned_filtered.csv"
image_base = "data/images/images"

df_cleaned = pd.read_csv(filtered_path)

# Build image paths
full_paths = [
    os.path.join(image_base, src, fn)
    for src, fn in zip(df_cleaned["media_source"], df_cleaned["image_fn"])
]

# Scan and collect broken paths again
broken_after_cleaning = [p for p in tqdm(full_paths, desc="🔍 Validating cleaned data") if not os.path.exists(p)]

# Report
if not broken_after_cleaning:
    print("✅ Validation passed: No broken image paths in filtered dataset.")
else:
    print(f"❌ {len(broken_after_cleaning)} broken image paths still exist.")
    print("🔧 Example:", broken_after_cleaning[:3])


🔍 Validating cleaned data: 100%|██████████████████████████████████████████████| 87971/87971 [00:20<00:00, 4280.89it/s]

✅ Validation passed: No broken image paths in filtered dataset.


In [12]:
class FakeNewsDataset(Dataset):
    def __init__(self, csv_path, image_root_dir, max_length=128):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.image_root = image_root_dir
        self.max_length = max_length

        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

        self.image_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        # --- Text (BERT input) ---
        title = str(row["title"])
        desc = str(row["description"])
        combined_text = title + " [SEP] " + desc

        encoding = self.tokenizer(
            combined_text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # --- Image (CNN input) ---
        img_path = os.path.join(self.image_root, row["media_source"], row["image_fn"])
        try:
            image = Image.open(img_path)
            if image.mode != "RGB":
                image = image.convert("RGB")
            image = self.image_transform(image)
        except:
            image = torch.zeros((3, 224, 224))  # fallback (should rarely happen now)

        # --- Label ---
        label = torch.tensor(row["is_fake"], dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "image": image,
            "label": label
        }


In [27]:
from torch.utils.data import DataLoader

dataset = FakeNewsDataset(
    csv_path="data/evons_cleaned_filtered.csv",
    image_root_dir="data/images/images"
)

dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)


FileNotFoundError: [Errno 2] No such file or directory: 'data/evons_cleaned_filtered.csv'

In [14]:
dataset = FakeNewsDataset(
    csv_path="data/evons_cleaned_filtered.csv",
    image_root_dir="data/images/images"
)

torch.save(dataset, "data/fake_news_dataset.pt")


In [12]:
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import os
from tqdm import tqdm

# -------------------------
# 1. Define your dataset class
# -------------------------
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, csv_path, image_root_dir, max_length=128):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.image_root = image_root_dir
        self.max_length = max_length
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
        self.image_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        title = str(row["title"])
        desc = str(row["description"])
        combined_text = title + " [SEP] " + desc

        encoding = self.tokenizer(
            combined_text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        img_path = os.path.join(self.image_root, row["media_source"], row["image_fn"])
        try:
            image = Image.open(img_path)
            if image.mode != "RGB":
                image = image.convert("RGB")
            image = self.image_transform(image)
        except:
            image = torch.zeros((3, 224, 224))

        label = torch.tensor(row["is_fake"], dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "image": image,
            "label": label
        }

In [13]:
class HybridModel(nn.Module):
    def __init__(self, num_classes=2):
        super(HybridModel, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.bert_fc = nn.Linear(self.bert.config.hidden_size, 256)

        self.cnn = models.resnet18(pretrained=True)
        self.cnn.fc = nn.Linear(self.cnn.fc.in_features, 256)

        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(512, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, input_ids, attention_mask, image):
        text_feat = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        text_feat = self.bert_fc(text_feat)
        image_feat = self.cnn(image)
        combined = torch.cat((text_feat, image_feat), dim=1)
        x = self.dropout(combined)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [14]:
csv_path = "C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/data/evons_cleaned_filtered.csv"
image_root = "data/images/images"
dataset = FakeNewsDataset(csv_path, image_root)
val_loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=2)


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridModel().to(device)
model.load_state_dict(torch.load("hybrid_model.pt", map_location=device))
model.eval()

C:\Users\Shreyash Bhamare\anaconda3\envs\FKN\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Shreyash Bhamare\anaconda3\envs\FKN\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


HybridModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        image = batch["image"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask, image)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

  0%|                                                                                         | 0/5499 [00:00<?, ?it/s]

In [5]:
print(len(val_loader))  # number of batches
print(len(val_loader.dataset))  # total samples


5499
87971


In [9]:
from torch.utils.data import Subset
from tqdm import tqdm
import torch

# ⚡ Only use the first 5000 samples for evaluation
subset_indices = list(range(min(5000, len(val_loader.dataset))))
val_subset = Subset(val_loader.dataset, subset_indices)
mini_val_loader = DataLoader(val_subset, batch_size=16, shuffle=False, num_workers=0)

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for i, batch in enumerate(tqdm(mini_val_loader, desc="⚡ Quick Eval")):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        image = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(input_ids, attention_mask, image)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        if i % 2 == 0:
            print(f"✅ Batch {i+1} processed")

print(f"\nTotal predictions: {len(all_preds)}")


⚡ Quick Eval:   0%|▏                                                                  | 1/313 [00:00<03:14,  1.61it/s]

✅ Batch 1 processed


⚡ Quick Eval:   1%|▋                                                                  | 3/313 [00:01<02:30,  2.06it/s]

✅ Batch 3 processed


⚡ Quick Eval:   2%|█                                                                  | 5/313 [00:02<02:08,  2.40it/s]

...

✅ Batch 309 processed


⚡ Quick Eval: 100%|████████████████████████████████████████████████████████████████▊| 312/313 [01:40<00:00,  7.14it/s]

✅ Batch 311 processed


⚡ Quick Eval: 100%|█████████████████████████████████████████████████████████████████| 313/313 [01:41<00:00,  3.10it/s]

✅ Batch 313 processed

Total predictions: 5000


In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your cleaned dataset
df = pd.read_csv("C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/data/evons_cleaned_filtered.csv")  # adjust path if needed

# Step 1: Train + Temp (30% reserved for val+test)
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df['is_fake'],
    random_state=42
)

# Step 2: Validation + Test (15% each from the full dataset)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['is_fake'],
    random_state=42
)

# Save to CSVs (optional but recommended)
train_df.to_csv("C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/data/evons_train.csv", index=False)
val_df.to_csv("C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/data/evons_val.csv", index=False)
test_df.to_csv("C:/Users/Shreyash Bhamare/Desktop/IIIT Pune/Projects/Fake News Detection/data/evons_test.csv", index=False)


# Confirm sizes
print(f"✅ Train set size: {len(train_df)}")
print(f"✅ Validation set size: {len(val_df)}")
print(f"✅ Test set size: {len(test_df)}")
print(f"🔍 Class balance in each split:")
print(train_df['is_fake'].value_counts(normalize=True))
print(val_df['is_fake'].value_counts(normalize=True))
print(test_df['is_fake'].value_counts(normalize=True))


✅ Train set size: 61579
✅ Validation set size: 13196
✅ Test set size: 13196
🔍 Class balance in each split:
is_fake
0    0.551811
1    0.448189
Name: proportion, dtype: float64
is_fake
0    0.551758
1    0.448242
Name: proportion, dtype: float64
is_fake
0    0.551834
1    0.448166
Name: proportion, dtype: float64


In [24]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import torch
import numpy as np
from tqdm import tqdm

def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="🔍 Evaluating"):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            image = batch["image"].to(device, non_blocking=True)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask, image)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    print("\n📊 Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    return {
        "accuracy": round(acc, 4),
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1_score": round(f1, 4)
    }


In [ ]:
metrics = evaluate_model(model, val_loader, device)

# Log or save later
print("\n🔧 Summary of Metrics:")
for k, v in metrics.items():
    print(f"{k.title()}: {v}")


🔍 Evaluating:   0%|                                                                          | 0/5499 [00:00<?, ?it/s]